In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv')


In [2]:
import gc
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer


In [3]:

train['is_train'] = 1
test['is_train'] = 0

test_temp = test.copy()
test_temp['label'] = np.nan 

all_data = pd.concat([train, test_temp], axis=0, ignore_index=True)

thread_stats = all_data.groupby('post_id').agg({
    'comment': 'count',              # How long is the thread? (Long threads = arguments)
    'upvote': ['mean', 'std', 'sum'], # Is the thread popular?
    'downvote': ['mean', 'std', 'sum'] # Is the thread hated/controversial?
})

thread_stats.columns = [
    'thread_vol', 
    'thread_up_mean', 'thread_up_std', 'thread_up_sum', 
    'thread_down_mean', 'thread_down_std', 'thread_down_sum'
]
thread_stats = thread_stats.reset_index()

thread_stats['thread_engagement'] = thread_stats['thread_up_sum'] + thread_stats['thread_down_sum']
thread_stats['thread_negativity_ratio'] = thread_stats['thread_down_sum'] / (thread_stats['thread_up_sum'] + 1)

train = train.merge(thread_stats, on='post_id', how='left')
test = test.merge(thread_stats, on='post_id', how='left')

fill_zero_cols = ['thread_up_std', 'thread_down_std']
train[fill_zero_cols] = train[fill_zero_cols].fillna(0)
test[fill_zero_cols] = test[fill_zero_cols].fillna(0)

print("Thread features added successfully.")

Thread features added successfully.


In [4]:
import re

def add_term_indicator_columns(
    df,
    terms,
    prefix,
    case_insensitive=True
):
    df = df.copy()

    flags = re.IGNORECASE if case_insensitive else 0

    for term in terms:
        col_name = f"{prefix}_{term}"
        pattern = re.escape(term)
        df[col_name] = df["comment"].str.contains(
            pattern,
            regex=True,
            flags=flags,
            na=False
        )

    return df


In [5]:
# === Feature Engineering and Preprocessing ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import FeatureUnion


thread_features = [
    'thread_vol', 'thread_up_mean', 'thread_up_std', 
    'thread_down_mean', 'thread_down_std', 
    'thread_engagement', 'thread_negativity_ratio'
]

# Update X_full to include these new columns
# Note: We added 'post_id' to the list just in case, but we won't use it in the model directly
cols_to_use = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 
    'upvote', 'downvote', "if_1", "if_2", 
    "disability", "race", "religion", "gender", 
    'comment'
] + thread_features 

X_full = train[cols_to_use].copy()
y = train['label']

X_full["text_len"] = X_full["comment"].str.len()
X_full["word_count"] = X_full["comment"].str.split().str.len()

X_full["question_marks"] = X_full["comment"].str.count(r"\?")
X_full["exclamations"] = X_full["comment"].str.count(r"!")

X_full[["text_len", "word_count"]]  = X_full[["text_len", "word_count"]].fillna(0)
X_full[["question_marks", "exclamations",]] = X_full[["question_marks", "exclamations"]].fillna(False)

X_full = X_full.fillna('Null')

violent_terms = [
    'kill', 'die', 'death', 'shoot', 'fire', 'killing', 'dead', 'hang', 'hit',
    'burn', 'murder', 'rid', 'shot', 'punch', 'shooting', 'guns', 'suicide',
    'bomb', 'gun', 'fired', 'kick', 'knife', 'killed', 'torture', 'bullet',
    'slap', 'destroy', 'stab', 'kill', 'killed', 'killer', 'murder', 'murderer',
    'suicide', 'nuke', 'pulverize', 'holocaust', 'dead', 'death', 'die',
    'gun', 'fire', 'shoot', 'bullet', 'missile', 'blow up', 'poison',
    'dumping body', 'strike fear', 'jail', 'prison', 'lock her up', 'prosecute',
    'death penalty', 'terrorist', 'meth dealer', 'war zone', 'sanctuary status'
]


X_full = add_term_indicator_columns(X_full,terms=violent_terms,prefix="v")

moral_terms = ["deserve", "should", "solution", "need to", "have to", "all of them"]
X_full = add_term_indicator_columns(X_full,terms=moral_terms,prefix="m")

group_terms = [
    'muslim', 'jew', 'jewish', 'christian', 'catholic', 'women', 'men',
    'female', 'male', 'black', 'white', 'asian', 'immigrant', 'refugee',
    'gay', 'lesbian', 'native', 'indian', 'muslim', 'catholic', 'jewish',
    'christian', 'buddhist', 'infidel', 'haram', 'devout', 'religious',
    'white', 'black', 'native american', 'ethnic', 'liberal', 'left winger',
    'progressive', 'theo-con', 'trump', 'blm', 'ku klux klan', 'neo-nazi',
    'white supremacist', 'woman', 'female', 'male', 'gay', 'homosexual',
    'refugee', 'asylum seeker'
              ]

X_full = add_term_indicator_columns(X_full,terms=group_terms,prefix="g")

personal_attack = [
    "lunatic", "looney", "mindless drivel", "ignorant", "ignorance",
    "stupid", "foolish", "gullible", "hallucination", "mental condition",
    "greedy", "criminal", "ethically challenged", "corruption", "rot",
    "cultist", "sheep", "loudmouth", "vampire", "traitor", 
    "troll", "arrogant", "liar", "lying", "weak", "victim complex",
    "mom's basement", "strawman", "hypocrite"
                  ]

X_full = add_term_indicator_columns(X_full,terms=personal_attack,prefix="p")


X_full["caps_ratio"] = X_full["comment"].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1))

test = test[cols_to_use].copy()
# y = train['label']

test["text_len"] = test["comment"].str.len()
test["word_count"] = test["comment"].str.split().str.len()

test["question_marks"] = test["comment"].str.count(r"\?")
test["exclamations"] = test["comment"].str.count(r"!")

test[["text_len", "word_count"]]  = test[["text_len", "word_count"]].fillna(0)
test[["question_marks", "exclamations",]] = test[["question_marks", "exclamations"]].fillna(False)
test["caps_ratio"] = test["comment"].apply(lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1))


test = test.fillna('Null')
test = add_term_indicator_columns(test,terms=violent_terms,prefix="v")
test = add_term_indicator_columns(test,terms=moral_terms,prefix="m")
test = add_term_indicator_columns(test,terms=group_terms,prefix="g")
test = add_term_indicator_columns(test,terms=personal_attack,prefix="p")




base_num_features = [
    'emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote', "if_1", "if_2", 
    "disability", "text_len", "word_count", "question_marks", "exclamations", "caps_ratio"
    ] + thread_features

term_feature_cols = [
    c for c in X_full.columns
    if c.startswith(("v_", "m_", "g_", "p_"))
]

num_features = base_num_features + term_feature_cols
# text_feature = "comment"
cat_features = ["race", "religion", "gender"]


feature_encoder = ColumnTransformer(
    transformers=[
        # ("text_vec", TfidfVectorizer(max_features=20000,ngram_range=(1, 2)), text_feature),
        ("num_scaler", StandardScaler(), num_features), 
        ("cats", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
)



In [6]:
# ------------------------------------------------------------------------------
print("Vectorizing Text (Max Parameters)...")

# Pipeline A: Word N-Grams (Context)
# Captures "kill you" vs "kill time"
word_vec = TfidfVectorizer(
    max_features=50000,     # Increased from 20k
    ngram_range=(1, 3),     # Uni, Bi, and Trigrams
    sublinear_tf=True,      # Log scaling for better robustness
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}'
)

# Pipeline B: Character N-Grams (Obfuscation/Robustness)
# Captures "k!ll", "sh1t", "f**k" without needing regex for everything
char_vec = TfidfVectorizer(
    max_features=50000,     # High dim for character chunks
    ngram_range=(3, 5),     # 3-5 character chunks
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char_wb'      # 'wb' creates word-boundary aware char ngrams
)

# Fit on ALL text (Train + Test) to ensure shared vocabulary
# (This is allowed and standard practice in Kaggle)
all_text = pd.concat([X_full['comment'], test['comment']])
word_vec.fit(all_text)
char_vec.fit(all_text)

# Transform Train and Test
train_word = word_vec.transform(X_full['comment'])
train_char = char_vec.transform(X_full['comment'])
test_word = word_vec.transform(test['comment'])
test_char = char_vec.transform(test['comment'])

# Combine Text Features
X_train_text = hstack([train_word, train_char])
X_test_text = hstack([test_word, test_char])

print(f"Text Feature Shape: {X_train_text.shape}")



Vectorizing Text (Max Parameters)...
Text Feature Shape: (198000, 100000)


In [34]:
from sklearn.feature_selection import SelectPercentile, chi2
from sklearn.decomposition import TruncatedSVD
import numpy as np

# Assuming train_tfidf, test_tfidf, and y_train are ready

print("Starting Phase 1: Supervised Feature Selection (Chi2)...")

selector = SelectPercentile(chi2, percentile=20)

# 2. Fit on training data and transform both
X_train_text_reduced = selector.fit_transform(X_train_text, y)
X_test_text_reduced = selector.transform(X_test_text)

print(f"Features reduced from {X_train_text.shape[1]} to {X_train_text_reduced.shape[1]}")

print("Starting Phase 2: Unsupervised Dimensionality Reduction (SVD)...")
# 2. Compress the high-signal filtered words into dense semantic concepts
# You can safely lower this now because the noise is gone
n_components = 200 
svd = TruncatedSVD(n_components=n_components, random_state=42)

train_text_dense = svd.fit_transform(X_train_text_reduced)
test_text_dense = svd.transform(X_test_text_reduced)

explained_variance = svd.explained_variance_ratio_.sum()
print(f"Variance explained by {n_components} components on filtered data: {explained_variance * 100:.2f}%")

# 3. Stack with metadata (if you have it) and train
# X_train_final = np.hstack([train_text_dense, train_metadata])
# X_test_final = np.hstack([test_text_dense, test_metadata])

# svd_model.fit(X_train_final, y_train)

Starting Phase 1: Supervised Feature Selection (Chi2)...
Features reduced from 100000 to 20000
Starting Phase 2: Unsupervised Dimensionality Reduction (SVD)...
Variance explained by 200 components on filtered data: 28.73%


In [10]:
import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD

# Assuming you have separated your data into:
# train_tfidf (sparse matrix of just the text features)
# test_tfidf (sparse matrix of just the text features)
# train_metadata (dense array/dataframe of upvotes, emoticons, etc.)
# test_metadata (dense array/dataframe of upvotes, emoticons, etc.)

# 1. Initialize Truncated SVD
# 100 to 300 components is the standard "sweet spot" for Kaggle NLP
n_components = 1000 
svd = TruncatedSVD(n_components=n_components, random_state=42)

print("Fitting SVD on training data...")
# 2. Fit and transform the training text data
# CRITICAL: Only fit on train to prevent data leakage from the test set!
train_text_dense = svd.fit_transform(X_train_text)

# 3. Transform the test text data using the fitted SVD
test_text_dense = svd.transform(X_test_text)

# Check how much variance is explained by these 150 components
explained_variance = svd.explained_variance_ratio_.sum()
print(f"Total variance explained by {n_components} components: {explained_variance * 100:.2f}%")



Fitting SVD on training data...
Total variance explained by 1000 components: 34.16%


In [17]:
train_text_dense.shape

(198000, 1000)

150 n_components explained 11.84% variance

In [35]:
# 4. Stack the dense SVD text features with your existing metadata
# Using np.hstack because the resulting matrices are now mostly dense

X_train_meta = feature_encoder.fit_transform(X_full)
X_test_meta = feature_encoder.transform(test)

X_full_train = np.hstack([train_text_dense, X_train_meta])
X_full_test = np.hstack([test_text_dense, X_test_meta])

print(f"Final training set shape: {X_full_train.shape}")

Final training set shape: (198000, 361)


In [ ]:
from sklearn.feature_selection import SelectPercentile, chi2
import numpy as np

print('starting selector ')

# 1. Initialize the selector
# We keep only the top 20% of the 45,000 features (approx 9,000 features)
# This usually retains ~98% of the model's predictive power.
selector = SelectPercentile(chi2, percentile=20)

print(f"Original shape: {X_train_text.shape}")

# 2. Fit on training data and transform both
X_train_text_reduced = selector.fit_transform(X_train_text, y)
X_test_text_reduced = selector.transform(X_test_text)

print(f"Reduced shape: {X_train_text_reduced.shape}")

In [ ]:
X_train_meta = feature_encoder.fit_transform(X_full)
X_test_meta = feature_encoder.transform(test)

# Combine EVERYTHING for LightGBM (Sparse + Dense)
X_full_train = hstack([X_train_text_reduced, X_train_meta]).tocsr()
X_full_test = hstack([X_test_text_reduced, X_test_meta]).tocsr()




In [36]:
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import time


X_train, X_valid, y_train, y_valid = train_test_split(
    X_full_train, y, test_size=0.2, random_state=0, stratify=y
)


modelA = lgb.LGBMClassifier(
        n_estimators=250,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=10,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.1,
        reg_alpha=0.5,
        reg_lambda=1.5,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        )



In [37]:
import lightgbm as lgb
from lightgbm import log_evaluation, early_stopping
import time

# 1. Setup Callbacks
# log_evaluation(100) will print progress every 100 iterations
# early_stopping(100) stops training if the validation score doesn't improve for 100 rounds
callbacks = [
    log_evaluation(period=100), 
    # early_stopping(stopping_rounds=100)
]

start_time = time.time()

# 2. Fit with progress trackinga
modelA.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],  # Tracking performance on validation set
    eval_metric='multi_logloss',    # Standard for multiclass progress tracking
    callbacks=callbacks
)

end_time = time.time()
training_time = end_time - start_time
print(f"\n--- Training Finished ---")
print(f"Total Training Time: {training_time/60:.2f} minutes.")


best_iter = modelA.best_iteration_
print(f"\nBest Iteration found: {best_iter}")


# 3. Final Evaluation
preds = modelA.predict(X_valid)
print(classification_report(y_valid, preds))

[100]	valid_0's multi_logloss: 0.453311
[200]	valid_0's multi_logloss: 0.366434

--- Training Finished ---
Total Training Time: 1.12 minutes.

Best Iteration found: 0


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

           0       0.97      0.94      0.95     22835
           1       0.78      0.70      0.74      3183
           2       0.81      0.91      0.86     12488
           3       0.73      0.28      0.40      1094

    accuracy                           0.90     39600
   macro avg       0.82      0.71      0.74     39600
weighted avg       0.90      0.90      0.89     39600



In [38]:
from scipy.optimize import minimize
from sklearn.metrics import f1_score, classification_report
import numpy as np

# ==========================================
# PHASE 4 (HYBRID): AGGRESSIVE OPTIMIZATION
# ==========================================

print("Generating validation probabilities...")
val_probs = modelA.predict_proba(X_valid) 

Generating validation probabilities...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [39]:
# 1. Define the Loss Function (Negative F1)
def f1_loss(weights):
    # Clip weights to be non-negative
    w = np.maximum(weights, 0) 
    weighted_probs = val_probs * w
    final_preds = np.argmax(weighted_probs, axis=1)
    return -f1_score(y_valid, final_preds, average='macro')

# 2. Define a list of diverse starting points (The "Candidates")
# We focus on variations around your successful manual logic
init_candidates = [
    # Baseline
    [1.0, 1.0, 1.0, 1.0],
    
    # Your manual "Sweet Spot" approximations
    [1.0, 1.0, 1.0, 4.0],  # ~0.25 threshold
    [1.0, 1.0, 1.0, 5.0],  # ~0.20 threshold
    [1.0, 1.0, 1.0, 6.0],  # ~0.16 threshold
    
    # High Aggression on Hate (1) and Violence (3)
    [1.0, 2, 1.2, 5.5],  
    [0.9, 1.3, 1.1, 4.5],
    
    # Suppress Civil (0) to force hard calls
    [0.8, 1.2, 1.2, 4.0],
    
    # "The Kitchen Sink" (High variance)
    [0.5, 2.0, 2.0, 8.0],

    #MANUAL TRY
    [1.0, 2.0, 1.2, 5.5],
    [1, 1.5, 1.2, 5],
    [1.0, 2.0, 1.5, 5],
    [1.0, 1.2, 1.0, 4],
    [0.8, 1.3, 1.1, 4.5],
]

best_global_score = -1
best_global_weights = None
best_start_point = None

# 3. The Loop (Try all starting points)
for i, init_w in enumerate(init_candidates):
    print(f"\n[Attempt {i+1}/{len(init_candidates)}] Starting from: {init_w}")
    
    # Run Optimizer
    opt_result = minimize(
        f1_loss, 
        init_w, 
        method='Powell', 
        tol=1e-5,
        options={'maxiter': 500}
    )
    
    current_score = -opt_result.fun
    current_weights = np.maximum(opt_result.x, 0) # Clip negative results
    
    print(f"  -> Result F1: {current_score:.5f}")
    
    # Update Winner
    if current_score > best_global_score:
        best_global_score = current_score
        best_global_weights = current_weights
        best_start_point = init_w
        print(f"  -> ⭐ NEW BEST FOUND!")

# 4. Compare with Pure Manual Threshold (Safety Check)
manual_preds = np.argmax(val_probs, axis=1)
manual_preds[val_probs[:, 3] > 0.2] = 3
manual_f1 = f1_score(y_valid, manual_preds, average='macro')

print("\n" + "="*40)
print(f"FINAL RESULTS")
print("="*40)
print(f"Manual Threshold (0.2) F1 : {manual_f1:.5f}")
print(f"Optimizer Best F1         : {best_global_score:.5f}")

if best_global_score > manual_f1:
    print("✅ Optimization Winner! Using optimized weights.")
    final_weights = best_global_weights
else:
    print("⚠️ Manual Threshold logic is simpler and better. Using Manual equivalent.")
    # Approximate the 0.2 threshold behavior with weights
    final_weights = np.array([1.0, 1.0, 1.0, 5.0]) 

print(f"Final Weights Chosen: {np.round(final_weights, 3)}")


[Attempt 1/13] Starting from: [1.0, 1.0, 1.0, 1.0]
  -> Result F1: 0.78295
  -> ⭐ NEW BEST FOUND!

[Attempt 2/13] Starting from: [1.0, 1.0, 1.0, 4.0]
  -> Result F1: 0.78309
  -> ⭐ NEW BEST FOUND!

[Attempt 3/13] Starting from: [1.0, 1.0, 1.0, 5.0]
  -> Result F1: 0.78184

[Attempt 4/13] Starting from: [1.0, 1.0, 1.0, 6.0]
  -> Result F1: 0.78225

[Attempt 5/13] Starting from: [1.0, 2, 1.2, 5.5]
  -> Result F1: 0.78316
  -> ⭐ NEW BEST FOUND!

[Attempt 6/13] Starting from: [0.9, 1.3, 1.1, 4.5]
  -> Result F1: 0.78335
  -> ⭐ NEW BEST FOUND!

[Attempt 7/13] Starting from: [0.8, 1.2, 1.2, 4.0]
  -> Result F1: 0.78345
  -> ⭐ NEW BEST FOUND!

[Attempt 8/13] Starting from: [0.5, 2.0, 2.0, 8.0]
  -> Result F1: 0.78283

[Attempt 9/13] Starting from: [1.0, 2.0, 1.2, 5.5]
  -> Result F1: 0.78316

[Attempt 10/13] Starting from: [1, 1.5, 1.2, 5]
  -> Result F1: 0.78356
  -> ⭐ NEW BEST FOUND!

[Attempt 11/13] Starting from: [1.0, 2.0, 1.5, 5]
  -> Result F1: 0.78308

[Attempt 12/13] Starting from: 

In [40]:
model_final = lgb.LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=10,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.1,
        reg_alpha=0.5,
        reg_lambda=1.5,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        )

# Fit on X_full_train (Combined Train + Val)

start_time = time.time()

# 2. Fit with progress tracking
model_final.fit(
    X_full_train, y,
    # eval_set=[(X_valid, y_valid)],  # Tracking performance on validation set
    # eval_metric='multi_logloss',    # Standard for multiclass progress tracking
)

end_time = time.time()
training_time = end_time - start_time
print(f"\n--- Training Finished ---")
print(f"Total Training Time: {training_time/60:.2f} minutes.")



--- Training Finished ---
Total Training Time: 2.88 minutes.


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import scipy.sparse as sp
from scipy.sparse import vstack

# Assume the following variables are already in your environment:
# X_train (sparse matrix of your pruned TF-IDF features + metadata)
# y_train (numpy array of training labels)
# X_test (sparse matrix of test features)
# best_lgb_model (your trained model that scores 0.83575)

# 1. Get probability distributions for the test set
test_probs = model_final.predict_proba(X_full_test)

# 2. Extract the max probability and the predicted class for each row
max_probs = np.max(test_probs, axis=1)
predicted_classes = np.argmax(test_probs, axis=1)

# 3. Define class-specific confidence thresholds
# You may need to tune these based on how many samples get extracted
thresholds = {
    0: 0.95,  # Civil (Easy class, demand high confidence)
    1: 0.90,  # Hate
    2: 0.90,  # Personal Attacks
    3: 0.85   # Violence (Hardest class, lower threshold to capture samples)
}

# Create a boolean mask for rows that meet their class's threshold
confident_mask = np.array([
    max_probs[i] >= thresholds[pred] 
    for i, pred in enumerate(predicted_classes)
])

# 4. Extract the confident test samples and their pseudo-labels
X_pseudo = X_full_test[confident_mask]
y_pseudo = predicted_classes[confident_mask]

print(f"Extracted {X_pseudo.shape[0]} confident pseudo-labels from the test set.")
unique, counts = np.unique(y_pseudo, return_counts=True)
print(f"Class distribution of pseudo-labels: {dict(zip(unique, counts))}")

# 5. Combine original training data with pseudo-labeled data
# Using vstack because X_train and X_pseudo are likely sparse matrices
X_train_combined = vstack([X_full_train, X_pseudo])
y_train_combined = np.concatenate([y, y_pseudo])

print(f"New training set shape: {X_train_combined.shape}")



# (Optional) Save to submission format
# submission_df = pd.read_csv('sample_submission.csv')
# submission_df['label'] = final_test_predictions
# submission_df.to_csv('pseudo_label_submission.csv', index=False)

In [ ]:
# (Optional) Save to submission format
submission_df = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv')



submission_df['label'] = final_test_predictions
submission_df.to_csv('pseudo_label_submission.csv', index=False)

In [ ]:
# import scipy.sparse as sp
# import numpy as np

# # Save the sparse matrix X
# sp.save_npz('pseudo_label_X_train_combined.npz', X_train_combined)

# # Save the labels y
# np.save('pseudo_label_y_train_combined.npy', y_train_combined)

# print("Saved successfully in native format!")

In [ ]:
# import scipy.sparse as sp
# import numpy as np

# # Load them right back into the exact same formats
# X_train_combined = sp.load_npz('pseudo_label_X_train_combined.npz')
# y_train_combined = np.load('pseudo_label_y_train_combined.npy')

In [ ]:
# 6. Retrain the model on the augmented dataset
# Use the exact same hyperparameters that got you your best score
pseudo_model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=10,
        min_child_samples=30,
        subsample=0.8,
        colsample_bytree=0.1,
        reg_alpha=0.5,
        reg_lambda=1.5,
        n_jobs=-1,
        random_state=0,
        verbosity=-1
        )

pseudo_model.fit(X_train_combined, y_train_combined)

# 7. Generate final predictions for the submission file
final_test_predictions = pseudo_model.predict(X_full_test)

In [30]:
final_weights = [1.607, 2.011, 1.799, 4.997]
final_weights = [1.339, 2.037, 1.007, 6.431]

In [41]:
test_probs = model_final.predict_proba(X_full_test)
weighted_test_probs = test_probs * final_weights
final_test_preds = np.argmax(weighted_test_probs, axis=1)

# Load sample submission to get correct format
sample_submission = pd.read_csv('/kaggle/input/competitions/comment-category-prediction-challenge/Sample.csv')

# Create submission dataframe
submission = pd.DataFrame({
    sample_submission.columns[0]: sample_submission.iloc[:, 0],  # ID column (first column)
    'label': final_test_preds  # Predictions
})

# Verify submission format
print("\nSubmission Preview:")
print(submission.head())
print(f"\nSubmission shape: {submission.shape}")
print(f"Prediction distribution:\n{submission['label'].value_counts().sort_index()}")

# Save submission file
submission.to_csv('submission_svd_200.csv', index=False)
print("\n✓ Submission file saved as 'submission.csv'")

# Verify no missing values
assert submission.isnull().sum().sum() == 0, "WARNING: Submission has missing values!"
print("✓ No missing values in submission")


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Submission Preview:
   ID  label
0   1      2
1   2      2
2   3      0
3   4      0
4   5      2

Submission shape: (102000, 2)
Prediction distribution:
label
0    57956
1     8527
2    33209
3     2308
Name: count, dtype: int64

✓ Submission file saved as 'submission.csv'
✓ No missing values in submission
